# Stage 0 pilot runner — Colab

Lightweight wrapper around `train.py`. Outputs persist to Drive so a session disconnect does not lose work.

Designed to work in two front-ends with the same notebook:
- **Native Colab (browser):** uses the clone cell to fetch the repo into `/content/repo`.
- **VS Code + Colab extension:** the local workspace is already synced; the clone cell becomes a no-op.

## Before running
1. Pro tier minimum is recommended (free tier idle disconnects make the suite painful).
2. The setup cell prints the allocated GPU type and measures throughput — use those numbers, not the project plan's GPU assumptions, to budget.
3. The suite is **idempotent**: re-running the round cell skips runs whose `eval.json` already exists, so reconnect after a disconnect and just re-run.

In [ ]:
# 1) Mount Drive (skip if running outside Colab — e.g. local Jupyter for a dry-run)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/concept_critic'
except ImportError:
    print('Not running on Colab — using local /tmp for outputs')
    DRIVE_ROOT = '/tmp/concept_critic'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('output root:', DRIVE_ROOT)

In [ ]:
# 2) Sync repo (idempotent across both front-ends)
#    - Native Colab: clones into /content/repo on first run, fetches+resets on later runs
#    - VS Code extension: if a local workspace is already mounted with train.py present, use it
import os, subprocess, sys

REPO_URL = 'https://github.com/AdeX11/concept_critic_models.git'
BRANCH   = 'domingo-experimental'   # pin a commit SHA for reproducibility, e.g. '3994e0e'

candidates = ['/content/repo', os.getcwd()]
REPO_DIR = None
for c in candidates:
    if os.path.exists(os.path.join(c, 'train.py')):
        REPO_DIR = c
        break

if REPO_DIR is None:
    REPO_DIR = '/content/repo'
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print('repo  :', REPO_DIR)
print('head  :', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())
print('branch:', subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip())

In [ ]:
# 3) Install deps + smoke + measure throughput on the GPU you actually got
!bash colab/setup.sh

In [ ]:
# 4) Round 1 — temporal architecture pilot (no prior results needed)
OUTPUT_DIR  = f'{DRIVE_ROOT}/stage0'
MAX_MINUTES = 660   # stay under Colab Pro session cap (~12h with margin)
BENCHMARKS  = 'armed_corridor phase_crossing'

!python colab/run_suite.py \
    --round round1 \
    --benchmarks $BENCHMARKS \
    --output_dir $OUTPUT_DIR \
    --max_minutes $MAX_MINUTES

## Resume after disconnect

Re-run the Round 1 cell. Already-completed runs are skipped (detected via `eval.json` presence in each run dir on Drive).

## Round 2+ (depend on prior round results)

Only run after the prior round is fully complete. `--results_root` is read to pick winners; `--output_dir` is where new runs land. Pointing them at the same Drive path is fine.

```bash
!python colab/run_suite.py \
    --round round2 \
    --benchmarks armed_corridor phase_crossing \
    --results_root $OUTPUT_DIR \
    --output_dir   $OUTPUT_DIR \
    --max_minutes  660
```

Same shape for `round3`, `round4`, `revalidate`, `confirm`, `ablation`.

## Aggregating results into a CSV

After any round, summarize all completed runs:

```bash
!python cluster/aggregate.py --results_root $OUTPUT_DIR --csv_out $OUTPUT_DIR/aggregate.csv
```

## Progress log

`run_suite.py` appends one JSON line per attempted run to `$OUTPUT_DIR/_suite_progress.jsonl` — useful for postmortem on which runs failed and why.